In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display
import warnings
warnings.filterwarnings('ignore')

# Parameters
BW = 125e3
fs = 2 * BW
SF_values = [7, 8, 9, 10, 11, 12]
SNR_dB = -15

# Fixed time axis based on SF12 (longest symbol)
max_SF = 12
max_M = 2**max_SF
max_Ts = max_M / BW
t_max_ms = max_Ts * 1e3

def generate_upchirp(SF, BW, fs, symbol_value=0):
    M = 2**SF
    Ts = M / BW
    N = int(fs * Ts)
    t = np.linspace(0, Ts, N, endpoint=False)
    f0 = -BW / 2
    k = BW / Ts
    shift = symbol_value / M
    upchirp = np.exp(1j * 2 * np.pi * (f0 * t + 0.5 * k * t**2 + shift * BW * t))
    return upchirp, t, Ts, N

def generate_downchirp(SF, BW, fs):
    M = 2**SF
    Ts = M / BW
    N = int(fs * Ts)
    t = np.linspace(0, Ts, N, endpoint=False)
    f0 = BW / 2
    k = -BW / Ts
    downchirp = np.exp(1j * 2 * np.pi * (f0 * t + 0.5 * k * t**2))
    return downchirp

def add_noise(signal, snr_db):
    signal_power = np.mean(np.abs(signal)**2)
    snr_linear = 10**(snr_db / 10)
    noise_power = signal_power / snr_linear
    noise = np.sqrt(noise_power / 2) * (np.random.randn(len(signal)) + 1j * np.random.randn(len(signal)))
    return signal + noise, noise

def dechirp_and_fft(signal, downchirp):
    dechirped = signal * downchirp
    fft_result = np.abs(np.fft.fft(dechirped))
    return fft_result

fig = plt.figure(figsize=(14, 10))
gs = fig.add_gridspec(3, 2, hspace=0.35, wspace=0.25)
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, 0])
ax4 = fig.add_subplot(gs[1, 1])
ax5 = fig.add_subplot(gs[2, 0])
ax6 = fig.add_subplot(gs[2, 1])

np.random.seed(42)

def animate(frame):
    SF = SF_values[frame % len(SF_values)]
    
    for ax in [ax1, ax2, ax3, ax4, ax5, ax6]:
        ax.clear()
    
    M = 2**SF
    Ts = M / BW
    symbol_value = M // 4
    
    upchirp, t, Ts, N = generate_upchirp(SF, BW, fs, symbol_value)
    downchirp = generate_downchirp(SF, BW, fs)
    noisy_signal, noise = add_noise(upchirp, SNR_dB)
    
    processing_gain_dB = 10 * np.log10(M)
    effective_snr = SNR_dB + processing_gain_dB
    
    t_ms = t * 1e3
    
    # Plot 1: Clean signal - FIXED X AXIS
    ax1.plot(t_ms, np.real(upchirp), 'b', linewidth=0.5)
    ax1.axvspan(0, Ts * 1e3, alpha=0.15, color='blue')
    ax1.set_title(f'Clean LoRa Chirp (SF={SF}, Ts={Ts*1e3:.1f}ms)', fontsize=11, fontweight='bold')
    ax1.set_xlabel('Time (ms)')
    ax1.set_ylabel('Amplitude')
    ax1.set_xlim([0, t_max_ms])
    ax1.set_ylim([-1.5, 1.5])
    ax1.grid(True, alpha=0.3)
    ax1.axvline(x=Ts * 1e3, color='blue', linestyle='--', alpha=0.7, linewidth=2)
    
    # Plot 2: Noisy signal - FIXED X AXIS
    ax2.plot(t_ms, np.real(noisy_signal), 'r', linewidth=0.3, alpha=0.7)
    ax2.axvspan(0, Ts * 1e3, alpha=0.15, color='red')
    ax2.set_title(f'Signal + Noise (SNR = {SNR_dB} dB)', fontsize=11, fontweight='bold')
    ax2.set_xlabel('Time (ms)')
    ax2.set_ylabel('Amplitude')
    ax2.set_xlim([0, t_max_ms])
    ax2.set_ylim([-8, 8])
    ax2.grid(True, alpha=0.3)
    ax2.axvline(x=Ts * 1e3, color='red', linestyle='--', alpha=0.7, linewidth=2)
    
    # Plot 3: Clean spectrogram - FIXED X AXIS
    max_N = int(fs * max_Ts)
    padded_upchirp = np.zeros(max_N, dtype=complex)
    padded_upchirp[:N] = upchirp
    
    nfft = 64
    ax3.specgram(padded_upchirp, Fs=fs, NFFT=nfft, noverlap=nfft-4, cmap='viridis')
    ax3.set_ylim([-BW/2, BW/2])
    ax3.set_xlim([0, max_Ts])
    ax3.axvline(x=Ts, color='white', linestyle='--', alpha=0.7, linewidth=2)
    ax3.set_title('Clean Chirp Spectrogram', fontsize=11)
    ax3.set_xlabel('Time (s)')
    ax3.set_ylabel('Frequency (Hz)')
    
    # Plot 4: Noisy spectrogram - FIXED X AXIS
    padded_noisy = np.zeros(max_N, dtype=complex)
    padded_noisy[:N] = noisy_signal
    noise_power = np.mean(np.abs(noise)**2)
    padded_noisy[N:] = np.sqrt(noise_power / 2) * (np.random.randn(max_N - N) + 1j * np.random.randn(max_N - N))
    
    ax4.specgram(padded_noisy, Fs=fs, NFFT=nfft, noverlap=nfft-4, cmap='viridis')
    ax4.set_ylim([-BW/2, BW/2])
    ax4.set_xlim([0, max_Ts])
    ax4.axvline(x=Ts, color='white', linestyle='--', alpha=0.7, linewidth=2)
    ax4.set_title('Noisy Signal Spectrogram', fontsize=11)
    ax4.set_xlabel('Time (s)')
    ax4.set_ylabel('Frequency (Hz)')
    
    # Plot 5: Dechirped FFT
    clean_fft = dechirp_and_fft(upchirp, downchirp)
    noisy_fft = dechirp_and_fft(noisy_signal, downchirp)
    
    num_bins = len(noisy_fft)
    bin_indices = np.arange(num_bins)
    
    clean_fft_norm = clean_fft / np.max(clean_fft)
    noisy_fft_norm = noisy_fft / np.max(noisy_fft)
    
    half = num_bins // 2
    peak_idx = np.argmax(noisy_fft_norm[:half])
    peak_val = noisy_fft_norm[peak_idx]
    
    mask = np.ones(half, dtype=bool)
    mask[max(0, peak_idx - 15):min(half, peak_idx + 15)] = False
    noise_floor = np.mean(noisy_fft_norm[:half][mask])
    
    zoom_width = int(half * 0.5)
    left_edge = max(0, peak_idx - zoom_width // 2)
    right_edge = min(half, peak_idx + zoom_width // 2)
    
    zoom_bins = bin_indices[left_edge:right_edge]
    zoom_noisy = noisy_fft_norm[left_edge:right_edge]
    zoom_clean = clean_fft_norm[left_edge:right_edge]
    
    ax5.fill_between(zoom_bins, 0, zoom_noisy, color='red', alpha=0.3)
    ax5.plot(zoom_bins, zoom_noisy, 'r-', linewidth=1.5, label='Noisy')
    ax5.plot(zoom_bins, zoom_clean, 'b--', linewidth=2, alpha=0.8, label='Clean')
    ax5.axhline(y=noise_floor, color='orange', linestyle='-', linewidth=2.5)
    ax5.plot(peak_idx, peak_val, 'go', markersize=15, markeredgecolor='black', markeredgewidth=2)
    
    ax5.set_title(f'Dechirped FFT - Peak at bin {peak_idx}', fontsize=11, fontweight='bold', color='darkgreen')
    ax5.set_xlabel('FFT Bin')
    ax5.set_ylabel('Magnitude')
    ax5.set_xlim([left_edge, right_edge])
    ax5.set_ylim([0, 1.4])
    ax5.legend(loc='upper right', fontsize=9)
    ax5.grid(True, alpha=0.3)
    
    # Plot 6: Processing gain comparison
    gains = [10 * np.log10(2**sf) for sf in SF_values]
    effective_snrs = [SNR_dB + g for g in gains]
    
    colors = ['limegreen' if sf == SF else 'lightgray' for sf in SF_values]
    edge_colors = ['darkgreen' if sf == SF else 'gray' for sf in SF_values]
    x_pos = np.arange(len(SF_values))
    
    bars = ax6.bar(x_pos, gains, color=colors, edgecolor=edge_colors, linewidth=2)
    ax6.axhline(y=-SNR_dB, color='red', linestyle='--', linewidth=2.5, label=f'Required: {-SNR_dB}dB')
    
    for i, (bar, eff_snr, sf) in enumerate(zip(bars, effective_snrs, SF_values)):
        color = 'green' if eff_snr > 0 else 'red'
        symbol = '✓' if eff_snr > 0 else '✗'
        ax6.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8, 
                f'{symbol}\n{eff_snr:.1f}dB', ha='center', va='bottom', fontsize=9, 
                color=color, fontweight='bold')
    
    ax6.set_xticks(x_pos)
    ax6.set_xticklabels([f'SF{sf}' for sf in SF_values], fontsize=10, fontweight='bold')
    ax6.set_ylabel('Processing Gain (dB)')
    ax6.set_title(f'Processing Gain | SF{SF} → {effective_snr:.1f}dB effective', fontsize=11, fontweight='bold')
    ax6.legend(loc='lower right', fontsize=9)
    ax6.grid(True, alpha=0.3, axis='y')
    ax6.set_ylim([0, 42])
    
    fig.suptitle(f'LoRa SF={SF} | Input SNR={SNR_dB}dB | Gain={processing_gain_dB:.1f}dB | Output={effective_snr:.1f}dB', 
                 fontsize=14, fontweight='bold', y=0.99)
    
    return [ax1, ax2, ax3, ax4, ax5, ax6]

anim = FuncAnimation(fig, animate, frames=len(SF_values), interval=2000, repeat=True)

gif_path = 'lora_noise_resistance.gif'
writer = PillowWriter(fps=0.5)
anim.save(gif_path, writer=writer, dpi=100)
plt.close()

print("Saved: lora_noise_resistance.gif")
print(f"\nFixed time axis: 0 to {t_max_ms:.1f} ms (SF12 duration)")
print("\nSF durations:")
for sf in SF_values:
    ts = (2**sf) / BW * 1000
    print(f"  SF{sf}: {ts:6.2f} ms")

display(Image(filename=gif_path))